# Compare node files: new run vs previous run

Compares the yearly node files (`yearly_node_files/nodes_start_*_end_*_year_*.csv.gz`) of two runs of the pipeline,
variable by variable. Run the cells from top to bottom; only **cell 1 (settings)** needs editing.

What you get:
1. **Schema**: columns added / removed / changed type
2. **Population**: rows, persons only in one run, active persons old vs new
3. **Column-by-column difference** for persons present in both runs (share identical / different / filled / emptied)
4. **Column permutation check**: detects mislabelled demographic columns (e.g. `gender` containing the migrant generation)
5. **Distributions** of a variable (or of all) old vs new
6. **Sanity checks**: value domains and formats, for both runs
7. **Plots** and optional export of the summary tables

All output is aggregated (counts, shares, means); no row-level data is printed. Check the CBS output rules before taking anything out of the RA.

**What to expect after the 2026-11 changes** (marked `CHG-2026-11` in the code):
- `active`, `gender`, `birth_year`, ... and the location columns (`buurt_code`, `wijk_code`, `gemeente_code`, `household_change_year`, buurt/gemeente metadata) differ slightly, because the address reference date is now 1 January (setting `address_reference` in `config.json`).
- If the old run had the positional column rename in script 02, `gender`, `migrant_generation` and `number_of_parents_from_abroad` can differ a lot. Section 4 shows this.
- `layers.csv` (typo `colleague`) does not appear in the node files.

## 1. Settings

In [ ]:
import os, re
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 300)

# ---- folders with the yearly node files of the two runs ----
OLD_DIR = r"H:\ODISSEI_portal_C_old\yearly_node_files"   # previous run
NEW_DIR = r"H:\ODISSEI_portal_C\yearly_node_files"       # new run

# ---- start / end year in the file names (nodes_start_{start}_end_{end}_year_{year}.csv.gz) ----
OLD_START, OLD_END = 2009, 2023
NEW_START, NEW_END = 2009, 2023
FILE_PATTERN = "nodes_start_{start}_end_{end}_year_{year}.csv.gz"

# ---- years to compare (start with one or two years; every year takes a while) ----
YEARS = [2015, 2022]

# ---- options ----
KEY = "label"                     # person identifier to join the two runs on
ONLY_ACTIVE_IN_BOTH = True        # compare values only for persons active in both runs (inactive rows are mostly empty)
FLOAT_TOLERANCE = 1e-9            # numbers closer than this count as identical
SHOW_EXTREMES = False             # min / max of numeric variables are individual values: only switch on if allowed
DEMOGRAPHIC_COLS = ["gender", "birth_year", "migrant_generation", "number_of_parents_from_abroad"]
OUT_DIR = None                    # e.g. r"H:\ODISSEI_portal_C\comparison" to save the summary tables as csv

## 2. Helpers (nothing to edit)

In [ ]:
def node_file(folder, start, end, year):
    return os.path.join(folder, FILE_PATTERN.format(start=start, end=end, year=year))

_loaded = {}
def load_pair(year):
    """Reads the old and the new node file of a year (the last year read is kept in memory)."""
    if year not in _loaded:
        _loaded.clear()
        old_fn, new_fn = node_file(OLD_DIR, OLD_START, OLD_END, year), node_file(NEW_DIR, NEW_START, NEW_END, year)
        print(f"Reading {year}:\n   old: {old_fn}\n   new: {new_fn}")
        _loaded[year] = (pl.read_csv(old_fn, infer_schema_length=100000),
                         pl.read_csv(new_fn, infer_schema_length=100000))
    return _loaded[year]

def as_bool(df, col="active"):
    """active as boolean, whatever way it was parsed from the csv."""
    dt = df.schema[col]
    if dt == pl.Boolean:
        return pl.col(col)
    if dt == pl.String:
        return pl.col(col).str.to_lowercase() == "true"
    return pl.col(col).cast(pl.Int64) != 0

def aligned(year):
    """Persons in both runs, old columns keep their names, new columns get the suffix _new."""
    old, new = load_pair(year)
    common = [c for c in new.columns if c in old.columns and c != KEY]
    j = old.select([KEY] + common).join(new.select([KEY] + common), on=KEY, how="inner", suffix="_new")
    if ONLY_ACTIVE_IN_BOTH and "active" in common:
        j = j.filter(as_bool(j, "active").fill_null(False) & as_bool(j, "active_new").fill_null(False))
    return j, common

def is_num(dtype):
    return dtype.is_numeric()

def save(df, name):
    if OUT_DIR:
        os.makedirs(OUT_DIR, exist_ok=True)
        df.to_csv(os.path.join(OUT_DIR, name), index=False)
        print("saved", os.path.join(OUT_DIR, name))

## 3. Schema: columns added / removed / changed type

In [ ]:
rows = []
for y in YEARS:
    old, new = load_pair(y)
    for c in sorted(set(old.columns) | set(new.columns)):
        rows.append(dict(year=y, column=c, in_old=c in old.columns, in_new=c in new.columns,
                         dtype_old=str(old.schema.get(c)), dtype_new=str(new.schema.get(c))))
schema_df = pd.DataFrame(rows)
changed = schema_df[(~schema_df.in_old) | (~schema_df.in_new) | (schema_df.dtype_old != schema_df.dtype_new)]
print("Columns with a difference in presence or dtype (empty = identical schemas):")
display(changed)
save(schema_df, "schema.csv")

## 4. Population: rows, persons only in one run, active persons

In [ ]:
rows = []
for y in YEARS:
    old, new = load_pair(y)
    r = dict(year=y, rows_old=old.height, rows_new=new.height,
             duplicate_labels_old=old.height - old[KEY].n_unique(), duplicate_labels_new=new.height - new[KEY].n_unique(),
             labels_only_old=old.select(KEY).join(new.select(KEY), on=KEY, how="anti").height,
             labels_only_new=new.select(KEY).join(old.select(KEY), on=KEY, how="anti").height)
    if "active" in old.columns and "active" in new.columns:
        a = old.select(KEY, as_bool(old).fill_null(False).alias("a_old")).join(
            new.select(KEY, as_bool(new).fill_null(False).alias("a_new")), on=KEY, how="inner")
        r.update(active_old=int(a["a_old"].sum()), active_new=int(a["a_new"].sum()),
                 active_both=int((a["a_old"] & a["a_new"]).sum()),
                 active_only_old=int((a["a_old"] & ~a["a_new"]).sum()),
                 active_only_new=int((~a["a_old"] & a["a_new"]).sum()))
    rows.append(r)
pop_df = pd.DataFrame(rows).set_index("year")
display(pop_df)
if "active_old" in pop_df:
    print("Share of the old active population that is no longer active, and of the new one that was not active before:")
    display(pd.DataFrame({"only_old_%": 100 * pop_df.active_only_old / pop_df.active_old,
                          "only_new_%": 100 * pop_df.active_only_new / pop_df.active_new}).round(3))
save(pop_df.reset_index(), "population.csv")

## 5. Column by column: how many values changed?

For persons in both runs (only those active in both runs if `ONLY_ACTIVE_IN_BOTH`):
`pct_diff` = share of persons with a value in both runs that is different; `pct_filled` = empty before, filled now; `pct_emptied` = filled before, empty now.

In [ ]:
def column_diff(year):
    j, common = aligned(year)
    n = j.height
    out = []
    for c in common:
        a, b = pl.col(c), pl.col(c + "_new")
        if is_num(j.schema[c]) and is_num(j.schema[c + "_new"]):
            a, b = a.cast(pl.Float64), b.cast(pl.Float64)
            same = ((a - b).abs() <= FLOAT_TOLERANCE).fill_null(False)
        else:
            a, b = a.cast(pl.String), b.cast(pl.String)
            same = a.eq_missing(b)
        s = j.select(
            both=(a.is_not_null() & b.is_not_null()).sum(),
            diff=(a.is_not_null() & b.is_not_null() & ~same).sum(),
            filled=(a.is_null() & b.is_not_null()).sum(),
            emptied=(a.is_not_null() & b.is_null()).sum(),
        ).row(0, named=True)
        out.append(dict(year=year, column=c, persons=n, both_filled=s["both"],
                        pct_diff=100 * s["diff"] / s["both"] if s["both"] else float("nan"),
                        pct_filled=100 * s["filled"] / n if n else float("nan"),
                        pct_emptied=100 * s["emptied"] / n if n else float("nan")))
    return pd.DataFrame(out)

diff_df = pd.concat([column_diff(y) for y in YEARS], ignore_index=True)
print("Variables with any difference (pct_diff, pct_filled or pct_emptied > 0):")
display(diff_df[(diff_df.pct_diff > 0) | (diff_df.pct_filled > 0) | (diff_df.pct_emptied > 0)].round(3).reset_index(drop=True))
print("Identical variables:", sorted(set(diff_df.column) - set(diff_df[(diff_df.pct_diff > 0) | (diff_df.pct_filled > 0) | (diff_df.pct_emptied > 0)].column)))
save(diff_df, "column_differences.csv")

## 6. Column permutation check (are demographic columns mislabelled?)

Share of persons for which old column (rows) equals new column (columns). A correct run shows ~100% on the diagonal.
A different pattern, e.g. `gender` (old) matching `migrant_generation` (new), means the old file had swapped columns.

In [ ]:
for y in YEARS:
    j, common = aligned(y)
    cols = [c for c in DEMOGRAPHIC_COLS if c in common]
    mat = pd.DataFrame(index=[f"old.{c}" for c in cols], columns=[f"new.{c}" for c in cols], dtype=float)
    for a in cols:
        for b in cols:
            e = j.select(((pl.col(a).cast(pl.Float64) == pl.col(b + "_new").cast(pl.Float64)).fill_null(False)).mean()).item()
            mat.loc[f"old.{a}", f"new.{b}"] = 100 * e
    print(f"Year {y}: % of persons with old column == new column")
    display(mat.round(1))

## 7. Distribution of a variable, old vs new

`compare_variable("gender", 2015)` for one variable. Numeric variables: share empty, mean, quartiles.
Categorical (or <= 30 distinct values): shares per value. Computed on active persons of each file separately.

In [ ]:
def compare_variable(col, year, active_only=True):
    old, new = load_pair(year)
    res = {}
    for name, df in (("old", old), ("new", new)):
        if col not in df.columns:
            print(f"{col} is not in the {name} file"); return
        if active_only and "active" in df.columns:
            df = df.filter(as_bool(df).fill_null(False))
        res[name] = df[col]
    s_old, s_new = res["old"], res["new"]
    if is_num(s_old.dtype) and is_num(s_new.dtype) and min(s_old.n_unique(), s_new.n_unique()) > 30:
        def summ(s):
            d = dict(persons=s.len(), pct_empty=100 * s.null_count() / s.len(), mean=s.mean(), std=s.std(),
                     q25=s.quantile(0.25), median=s.median(), q75=s.quantile(0.75))
            if SHOW_EXTREMES:
                d.update(min=s.min(), max=s.max())
            return d
        t = pd.DataFrame({"old": summ(s_old), "new": summ(s_new)})
        t["new-old"] = t["new"] - t["old"]
        print(f"{col}, {year}"); display(t.round(3))
    else:
        def shares(s):
            vc = s.cast(pl.String).fill_null("<empty>").value_counts()
            return pd.Series(dict(zip(vc[vc.columns[0]].to_list(), (vc["count"] / s.len() * 100).to_list())))
        t = pd.concat([shares(s_old).rename("old_%"), shares(s_new).rename("new_%")], axis=1).fillna(0)
        t["new-old"] = t["new_%"] - t["old_%"]
        print(f"{col}, {year}"); display(t.sort_values("old_%", ascending=False).head(40).round(3))

# one variable:
compare_variable("gender", YEARS[0])

In [ ]:
# all variables of one year (long output)
y = YEARS[0]
old, new = load_pair(y)
for c in [c for c in new.columns if c in old.columns and c not in (KEY, "id")]:
    compare_variable(c, y)

## 8. Sanity checks (value domains and formats)

Number of violating persons per check, for the old and the new file (active persons unless stated). `n/a` = column not in the file.

In [ ]:
def n_not_in(col, allowed):
    return lambda d, y: d.filter(as_bool(d).fill_null(False) & pl.col(col).is_not_null() & ~pl.col(col).is_in(allowed)).height
def n_regex_bad(col, pattern):
    return lambda d, y: d.filter(pl.col(col).is_not_null() & ~pl.col(col).cast(pl.String).str.contains(pattern)).height
def n_out_of_range(col, lo, hi):
    return lambda d, y: d.filter(as_bool(d).fill_null(False) & pl.col(col).is_not_null() &
                                 ((pl.col(col) < lo(y)) | (pl.col(col) > hi(y)))).height

CHECKS = [  # (check, columns needed, function(df, year) -> number of violations)
    ("label unique",                       ["label"],  lambda d, y: d.height - d["label"].n_unique()),
    ("id unique",                          ["id"],     lambda d, y: d.height - d["id"].n_unique()),
    ("id runs 0..N-1 (same ids in all years)", ["id"], lambda d, y: int(not (d["id"].min() == 0 and d["id"].max() == d.height - 1))),
    ("active has no empty values",         ["active"], lambda d, y: d["active"].null_count()),
    ("gender in {1,2}",                    ["active", "gender"], n_not_in("gender", [1, 2])),
    ("migrant_generation in {0,1,2}",      ["active", "migrant_generation"], n_not_in("migrant_generation", [0, 1, 2])),
    ("number_of_parents_from_abroad in {0,1,2}", ["active", "number_of_parents_from_abroad"], n_not_in("number_of_parents_from_abroad", [0, 1, 2])),
    ("missing_mother in {0,1}",            ["active", "missing_mother"], n_not_in("missing_mother", [0, 1])),
    ("missing_father in {0,1}",            ["active", "missing_father"], n_not_in("missing_father", [0, 1])),
    ("birth_year between 1850 and year",   ["active", "birth_year"], n_out_of_range("birth_year", lambda y: 1850, lambda y: y)),
    ("active persons without gender",      ["active", "gender"], lambda d, y: d.filter(as_bool(d).fill_null(False) & pl.col("gender").is_null()).height),
    ("inactive persons with a birth year", ["active", "birth_year"], lambda d, y: d.filter(~as_bool(d).fill_null(False) & pl.col("birth_year").is_not_null()).height),
    ("household_income_percentile in 1..100", ["active", "household_income_percentile"], n_out_of_range("household_income_percentile", lambda y: 1, lambda y: 100)),
    ("household_income >= -1",             ["active", "household_income"], n_out_of_range("household_income", lambda y: -1, lambda y: 10**12)),
    ("individual_income_percentile in 1..100 (if filled)", ["active", "individual_income_percentile"], n_out_of_range("individual_income_percentile", lambda y: 1, lambda y: 100)),
    ("buurt_code like BU00000000",         ["buurt_code"],   n_regex_bad("buurt_code", r"^BU\d{8}$")),
    ("wijk_code like WK000000",            ["wijk_code"],    n_regex_bad("wijk_code", r"^WK\d{6}$")),
    ("gemeente_code like GM0000",          ["gemeente_code"], n_regex_bad("gemeente_code", r"^GM\d{4}$")),
    ("household_change_year <= year",      ["household_change_year"], lambda d, y: d.filter(pl.col("household_change_year") > y).height),
]

rows = []
for y in YEARS:
    old, new = load_pair(y)
    for name, needed, fn in CHECKS:
        r = dict(year=y, check=name)
        for label, df in (("old", old), ("new", new)):
            r[label] = fn(df, y) if all(c in df.columns for c in needed) else None
        rows.append(r)
checks_df = pd.DataFrame(rows)
checks_df["status"] = checks_df.apply(lambda r: "n/a" if pd.isna(r.new) else ("OK" if r.new == 0 else "FAIL (new)"), axis=1)
print("Checks with violations in the new file, or not applicable:")
display(checks_df[checks_df.status != "OK"])
print(f"{(checks_df.status == 'OK').sum()} of {len(checks_df)} checks passed in the new file.")
save(checks_df, "sanity_checks.csv")

## 9. Plots

In [ ]:
if len(diff_df):
    piv = diff_df.pivot(index="column", columns="year", values="pct_diff")
    fig, ax = plt.subplots(figsize=(1.2 * len(YEARS) + 4, 0.3 * len(piv) + 2))
    im = ax.imshow(piv.values, aspect="auto", cmap="viridis", vmin=0, vmax=max(1, float(piv.max().max())))
    ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns)
    ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index)
    plt.colorbar(im, label="% of persons with a different value")
    ax.set_title("Share of changed values per variable (persons in both runs)")
    plt.tight_layout(); plt.show()

if "active_old" in pop_df:
    ax = pop_df[["active_old", "active_new"]].plot(marker="o", figsize=(7, 3.5), title="Active persons, old vs new")
    ax.set_ylabel("persons"); plt.tight_layout(); plt.show()